# From RGB Pixels to Plot-Level Crop Traits

## Self-paced RGB orthomosaic and Green Leaf Index workflow

This notebook is a **sequential workflow**. Run each cell once, from top to bottom. Objects created in one cell are reused in later cells, so GLI, the Otsu threshold, and the vegetation mask are not recalculated unnecessarily.

### Workflow

1. Load the RGB orthomosaic and plot boundaries.
2. Inspect raster metadata.
3. Name and display Red, Green, and Blue bands.
4. Calculate GLI once.
5. Calculate the Otsu threshold once.
6. Create the binary vegetation mask once.
7. Mask the RGB orthomosaic and GLI raster.
8. Overlay plot polygons interactively using `mapview() %>% leafem:::addGeoRaster()`.
9. Extract plot-level mean GLI, GLI variability, and canopy cover.
10. Export the plot-level trait table.

> Expected RGB band order: **Red, Green, Blue**. Verify the band order from the orthomosaic metadata before using another dataset.


## 1. Load packages and data

Run this cell first. The notebook expects the data files shown below.

```text
data/rgb_orthomosaic.tif
data/plot_boundaries.geojson
```


In [ ]:
library(terra)
library(sf)
library(mapview)
library(leafem)
library(magrittr)
library(DT)

terraOptions(progress = 0)
mapviewOptions(fgb = FALSE)

orthomosaic_path <- "data/rgb_orthomosaic.tif"
plots_path <- "data/plot_boundaries.geojson"

stopifnot(file.exists(orthomosaic_path))
stopifnot(file.exists(plots_path))

rgb <- terra::rast(orthomosaic_path)
plots <- terra::vect(plots_path)

if (terra::nlyr(rgb) < 3) {
  stop("The RGB orthomosaic must contain at least three bands.")
}

rgb <- rgb[[1:3]]
names(rgb) <- c("Red", "Green", "Blue")

plot_id_candidates <- c("plot_id", "Plot_ID", "plot", "Plot", "id", "ID")
plot_id_matches <- plot_id_candidates[plot_id_candidates %in% names(plots)]

if (length(plot_id_matches) == 0) {
  plots$plot_id <- paste0("Plot_", seq_len(nrow(plots)))
  plot_id_field <- "plot_id"
} else {
  plot_id_field <- plot_id_matches[[1]]
}

cat("Loaded RGB bands:", paste(names(rgb), collapse = ", "), "\n")
cat("Number of plots:", nrow(plots), "\n")


## 2. Inspect the RGB orthomosaic

Examine the number of layers, dimensions, spatial resolution, extent, and coordinate reference system.


In [ ]:
print(rgb)
cat("Bands:", terra::nlyr(rgb), "\n")
cat("Rows and columns:", terra::nrow(rgb), "x", terra::ncol(rgb), "\n")
cat("Resolution:", paste(terra::res(rgb), collapse = " x "), "\n")
cat(
  "Extent:",
  paste(
    c(terra::xmin(rgb), terra::xmax(rgb), terra::ymin(rgb), terra::ymax(rgb)),
    collapse = ", "
  ),
  "\n"
)
cat("CRS:", terra::crs(rgb), "\n")


## 3. Compare RGB bands and display the true-color composite

The individual layers show the intensity recorded in each visible band.


In [ ]:
par(mfrow = c(1, 3))
plot(rgb[["Red"]], col = hcl.colors(100, "Reds"), main = "Red band")
plot(rgb[["Green"]], col = hcl.colors(100, "Greens"), main = "Green band")
plot(rgb[["Blue"]], col = hcl.colors(100, "Blues"), main = "Blue band")
par(mfrow = c(1, 1))

terra::plotRGB(
  rgb,
  r = 1,
  g = 2,
  b = 3,
  stretch = "lin",
  main = "RGB True-Color Orthomosaic"
)


## 4. Calculate the Green Leaf Index once

The Green Leaf Index is:

\[
GLI = \frac{2Green - Red - Blue}{2Green + Red + Blue}
\]

This cell creates the `gli` object. Later cells reuse this object.


In [ ]:
red <- rgb[["Red"]]
green <- rgb[["Green"]]
blue <- rgb[["Blue"]]

gli_denominator <- 2 * green + red + blue

gli <- terra::ifel(
  gli_denominator == 0,
  NA,
  (2 * green - red - blue) / gli_denominator
)

names(gli) <- "GLI"

plot(
  gli,
  col = hcl.colors(100, "RdYlGn"),
  main = "Green Leaf Index"
)

terra::global(gli, c("min", "mean", "max"), na.rm = TRUE)


## 5. Calculate the Otsu threshold once

Otsu thresholding divides the GLI distribution into background and vegetation classes by maximizing between-class variance. This cell creates `otsu_threshold`.


In [ ]:
calculate_otsu_threshold <- function(raster, sample_size = 100000, bins = 256) {
  sampled <- terra::spatSample(
    raster,
    size = min(sample_size, terra::ncell(raster)),
    method = "regular",
    na.rm = TRUE,
    values = TRUE
  )

  pixel_values <- as.numeric(sampled[, 1])
  pixel_values <- pixel_values[is.finite(pixel_values)]

  if (length(pixel_values) < 2) {
    stop("Not enough valid GLI pixels for Otsu thresholding.")
  }

  h <- hist(pixel_values, breaks = bins, plot = FALSE)
  probabilities <- h$counts / sum(h$counts)
  cumulative_probability <- cumsum(probabilities)
  cumulative_mean <- cumsum(probabilities * h$mids)
  total_mean <- sum(probabilities * h$mids)

  between_class_variance <-
    (total_mean * cumulative_probability - cumulative_mean)^2 /
    (cumulative_probability * (1 - cumulative_probability))

  between_class_variance[!is.finite(between_class_variance)] <- NA_real_
  h$mids[which.max(between_class_variance)]
}

otsu_threshold <- calculate_otsu_threshold(gli)
cat("Otsu GLI threshold:", round(otsu_threshold, 4), "\n")

gli_values <- terra::values(gli, mat = FALSE)
gli_values <- gli_values[is.finite(gli_values)]

hist(
  gli_values,
  breaks = 80,
  col = "grey80",
  border = "white",
  main = "GLI Distribution and Otsu Threshold",
  xlab = "GLI"
)
abline(v = otsu_threshold, col = "red", lwd = 3, lty = 2)


## 6. Create the binary vegetation mask once

Pixels with GLI greater than the Otsu threshold are classified as vegetation.


In [ ]:
vegetation_mask <- terra::ifel(gli > otsu_threshold, 1, 0)
names(vegetation_mask) <- "Vegetation_Mask"

plot(
  vegetation_mask,
  col = c("grey85", "forestgreen"),
  main = "Binary Vegetation Mask"
)

cat("0 = non-vegetation; 1 = vegetation\n")


## 7. Mask the RGB orthomosaic and GLI layer

The existing `vegetation_mask` is reused. Background cells become `NA`.


In [ ]:
vegetation_only_mask <- terra::ifel(vegetation_mask == 1, 1, NA)

vegetation_rgb <- rgb * vegetation_only_mask
vegetation_gli <- terra::ifel(vegetation_mask == 1, gli, NA)
names(vegetation_gli) <- "Vegetation_GLI"

terra::plotRGB(
  vegetation_rgb,
  r = 1,
  g = 2,
  b = 3,
  stretch = "lin",
  main = "Vegetation-Only RGB"
)

plot(
  vegetation_gli,
  col = hcl.colors(100, "YlGn"),
  main = "Vegetation-Only GLI"
)


## 8. Align plot polygons

Project plot polygons to the raster coordinate system only when necessary.


In [ ]:
if (is.na(terra::crs(rgb, proj = TRUE)) || !nzchar(terra::crs(rgb, proj = TRUE))) {
  stop("The RGB orthomosaic does not contain a valid coordinate reference system.")
}

if (is.na(terra::crs(plots, proj = TRUE)) || !nzchar(terra::crs(plots, proj = TRUE))) {
  stop("The plot boundary layer does not contain a valid coordinate reference system.")
}

if (!terra::same.crs(vegetation_gli, plots)) {
  plots <- terra::project(plots, terra::crs(vegetation_gli))
}

plot(
  vegetation_gli,
  col = hcl.colors(100, "YlGn"),
  main = "Vegetation GLI with Plot Boundaries"
)
plot(plots, add = TRUE, border = "black", lwd = 1.5)


## 9. Interactive vector and raster overlay

This is the only interactive map. The map is created with `mapview()` and `leafem:::addGeoRaster()`. Use zoom, pan, layer controls, and plot labels.

`addGeoRaster()` is accessed with `:::` because it is an internal `leafem` function. Internal functions can change between package versions.


In [ ]:
plots_sf <- sf::st_as_sf(plots)
plots_wgs84 <- sf::st_transform(plots_sf, 4326)

# Start with vector plot boundaries.
plot_map <- mapview::mapview(
  plots_wgs84,
  zcol = NULL,
  color = "black",
  lwd = 2,
  alpha.regions = 0,
  layer.name = "Plot boundaries",
  label = as.character(plots_wgs84[[plot_id_field]])
)

# Add the vegetation-only GLI raster to the mapview object.
plot_map %>%
  leafem:::addGeoRaster(
    x = vegetation_gli,
    group = "Vegetation GLI",
    opacity = 0.90,
    resolution = 128,
    project = TRUE,
    autozoom = TRUE,
    colorOptions = leafem::colorOptions(
      palette = hcl.colors(100, "YlGn"),
      na.color = "transparent"
    )
  ) %>%
  leaflet::addLayersControl(
    overlayGroups = c("Vegetation GLI", "Plot boundaries"),
    options = leaflet::layersControlOptions(collapsed = FALSE)
  )


## 10. Extract plot-level crop traits

This cell uses the existing `vegetation_gli`, `vegetation_mask`, and `plots` objects. Nothing is recalculated.


In [ ]:
mean_result <- terra::extract(
  vegetation_gli,
  plots,
  fun = mean,
  na.rm = TRUE,
  exact = TRUE
)

sd_result <- terra::extract(
  vegetation_gli,
  plots,
  fun = sd,
  na.rm = TRUE,
  exact = TRUE
)

cover_result <- terra::extract(
  vegetation_mask,
  plots,
  fun = mean,
  na.rm = TRUE,
  exact = TRUE
)

plot_level_traits <- data.frame(
  plot_id = as.data.frame(plots)[[plot_id_field]],
  mean_vegetation_gli = round(mean_result[, 2], 4),
  sd_vegetation_gli = round(sd_result[, 2], 4),
  canopy_cover_percent = round(100 * cover_result[, 2], 2)
)

plot_level_traits


## 11. Inspect and export the trait table


In [ ]:
DT::datatable(
  plot_level_traits,
  rownames = FALSE,
  filter = "top",
  options = list(pageLength = 10, scrollX = TRUE)
)

output_file <- "outputs/plot_level_rgb_gli_traits.csv"
dir.create("outputs", showWarnings = FALSE)
write.csv(plot_level_traits, output_file, row.names = FALSE, na = "")
cat("Saved:", output_file, "\n")


## Workflow complete

You created a cumulative pixel-to-plot analysis without repeating earlier calculations:

```text
rgb → gli → otsu_threshold → vegetation_mask → vegetation_gli → plot_level_traits
```

Restart the kernel and use **Run All Cells** to repeat the complete workflow from the beginning.
